# MAIA - FNLP - Proyecto FACULTADES
Clasificación multiclase: dado el resumen de un trabajo académico en español, predecir a cuál de las 5 facultades pertenece.

**Clases:** `administracion_empresas` · `medicina` · `gobierno` · `derecho` · `economia`  
**Métrica:** F1-macro (sin ponderar por clase)  
**Reglas:** solo PLN clásico con scikit-learn · no transformers · Word2Vec/FastText permitidos · test.csv solo para predicciones finales

## 0. Configuración del entorno

In [ ]:
# Clona el repositorio
!git clone https://github.com/luzdvanegasp/kaggle-competition_FNLP.git
%cd kaggle-competition_FNLP

In [ ]:
!pip install -q nltk unidecode

## 1. Carga de librerías

### 1.1 Visualización y análisis de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

### 1.2 Preprocesamiento de texto

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

### 1.3 Vectorización

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

### 1.4 Modelos de clasificación

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.ensemble import VotingClassifier
from sklearn.calibration import CalibratedClassifierCV

### 1.5 Evaluación y ajuste de hiperparámetros

In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import time

### 1.6 Configuración general y carga de datos

In [ ]:
seed = 0
np.random.seed(seed)

train_path = 'inputs/train.csv'
val_path   = 'inputs/val.csv'
test_path  = 'inputs/test.csv'
text_col   = 'resumen'
label_col  = 'facultad'
id_col     = 'id'

train = pd.read_csv(train_path)
val   = pd.read_csv(val_path)
test  = pd.read_csv(test_path)

print(f'Train: {train.shape} | Val: {val.shape} | Test: {test.shape}')
train.head(2)

## 2. Análisis exploratorio (EDA)

In [ ]:
# Distribución de clases
fig, axes = plt.subplots(1, 2, figsize=(10, 6))
for ax, df, titulo in zip(axes, [train, val], ['Train (6.000)', 'Val (1.500)']):
    conteo = df[label_col].value_counts()
    conteo.plot(kind='bar', ax=ax, color='navy', edgecolor='white')
    ax.set_title(titulo)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height())}', (p.get_x()+p.get_width()/2, p.get_height()+5), ha='center', fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# Longitud de resúmenes
train['n_palabras'] = train[text_col].str.split().str.len()
plt.figure(figsize=(10, 4))
for label in sorted(train[label_col].unique()):
    subset = train[train[label_col]==label]['n_palabras']
    subset.plot(kind='kde', label=f'{label} (med={int(subset.median())})', color='navy' if label == sorted(train[label_col].unique())[0] else None)
plt.legend(fontsize=9)
plt.title('Distribución de longitud por facultad')
plt.xlabel('N° palabras'); plt.tight_layout(); plt.show()
print(train.groupby(label_col)['n_palabras'].describe()[['mean','min','50%','max']].round(0))

## 3. Preprocesamiento de texto

In [ ]:
stopwords_es = set(stopwords.words('spanish'))
extra_stop = {
    'objetivo','objetivos','presente','trabajo','tesis','estudio','investigacion',
    'resultado','resultados','conclusion','conclusiones','metodologia','analisis',
    'mediante','utilizando','utilizar','así','siendo','segun','dicho','dichos','dicha','cual','cuales'
}
stopwords_es.update(extra_stop)

def preprocess(text: str, quitar_sw: bool = True) -> str:
    if not isinstance(text, str): return ''
    text = text.lower()
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^a-záéíóúüñ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if quitar_sw:
        tokens = [t for t in text.split() if t not in stopwords_es and len(t) > 2]
        text = ' '.join(tokens)
    return text

# Aplicar solo a train y val (NUNCA a test antes de predecir)
train['texto_proc'] = train[text_col].apply(preprocess)
val['texto_proc']   = val[text_col].apply(preprocess)

print('Original :', train[text_col].iloc[0][:200])
print()
print('Procesado:', train['texto_proc'].iloc[0][:200])

## 4. Baseline — TF-IDF + comparativa de clasificadores

In [ ]:
x_train = train['texto_proc'].values
y_train = train[label_col].values
x_val   = val['texto_proc'].values
y_val   = val[label_col].values

tfidf_params = dict(ngram_range=(1,2), max_features=80_000, sublinear_tf=True, min_df=2, max_df=0.95)

modelos = {
    'LogReg C=1' : LogisticRegression(C=1,  max_iter=1000, class_weight='balanced', solver='lbfgs', multi_class='multinomial', random_state=seed),
    'LogReg C=5' : LogisticRegression(C=5,  max_iter=1000, class_weight='balanced', solver='lbfgs', multi_class='multinomial', random_state=seed),
    'LogReg C=10': LogisticRegression(C=10, max_iter=1000, class_weight='balanced', solver='lbfgs', multi_class='multinomial', random_state=seed),
    'LinearSVC'  : LinearSVC(C=1, class_weight='balanced', max_iter=3000, random_state=seed),
    'ComplementNB': ComplementNB(alpha=0.1),
}

resultados = {}
print(f"{'Modelo':<14} {'F1-macro val':>13} {'Tiempo':>8}")
print('-'*38)
for nombre, clf in modelos.items():
    pipe = Pipeline([('tfidf', TfidfVectorizer(**tfidf_params)), ('clf', clf)])
    t0 = time.time()
    pipe.fit(x_train, y_train)
    f1 = f1_score(y_val, pipe.predict(x_val), average='macro')
    resultados[nombre] = (f1, pipe)
    print(f"{nombre:<14} {f1:>13.4f} {time.time()-t0:>7.1f}s")

In [ ]:
# Comparación visual
nombres = list(resultados.keys())
f1s     = [resultados[n][0] for n in nombres]

plt.figure(figsize=(8, 4))
bars = plt.barh(nombres, f1s, color='navy', edgecolor='white')
for bar, val_f1 in zip(bars, f1s):
    plt.text(val_f1 + 0.002, bar.get_y() + bar.get_height()/2, f'{val_f1:.4f}', va='center', fontsize=9)
plt.xlabel('F1-macro val')
plt.title('Comparación de modelos baseline')
plt.xlim(0.7, 0.95)
plt.tight_layout(); plt.show()

In [ ]:
mejor_nombre = max(resultados, key=lambda k: resultados[k][0])
mejor_pipe   = resultados[mejor_nombre][1]
print(f'Mejor modelo: {mejor_nombre}  ->  F1-macro val = {resultados[mejor_nombre][0]:.4f}\n')
print(classification_report(y_val, mejor_pipe.predict(x_val)))

## 5. TF-IDF palabras + caracteres

In [ ]:
tfidf_word = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=80_000, sublinear_tf=True, min_df=2, max_df=0.95)
tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), max_features=50_000, sublinear_tf=True, min_df=3)

x_tr_combo = hstack([tfidf_word.fit_transform(x_train), tfidf_char.fit_transform(x_train)])
x_va_combo = hstack([tfidf_word.transform(x_val),       tfidf_char.transform(x_val)])

clf_combo = LogisticRegression(C=5, max_iter=1000, class_weight='balanced', solver='lbfgs', multi_class='multinomial', random_state=seed)
clf_combo.fit(x_tr_combo, y_train)
f1_combo = f1_score(y_val, clf_combo.predict(x_va_combo), average='macro')
print(f'TF-IDF word+char  ->  F1-macro val = {f1_combo:.4f}')

## 6. Ajuste de hiperparámetros (GridSearch sobre train+val)

In [ ]:
x_all = np.concatenate([x_train, x_val])
y_all = np.concatenate([y_train, y_val])

pipe_gs = Pipeline([
    ('tfidf', TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.95)),
    ('clf',   LogisticRegression(class_weight='balanced', max_iter=2000, solver='lbfgs', multi_class='multinomial', random_state=seed))
])

param_grid = {
    'tfidf__ngram_range':  [(1,1),(1,2),(1,3)],
    'tfidf__max_features': [60_000, 100_000, 150_000],
    'clf__C':              [1, 3, 5, 10],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
gs = GridSearchCV(pipe_gs, param_grid, cv=cv, scoring='f1_macro', n_jobs=-1, verbose=1)
gs.fit(x_all, y_all)

print(f'\nMejores parametros : {gs.best_params_}')
print(f'Mejor F1-macro CV  : {gs.best_score_:.4f}')

## 7. Análisis de errores

In [ ]:
# Reentrenamos el mejor pipeline solo con train para evaluar en val
gs.best_estimator_.fit(x_train, y_train)
y_pred_val = gs.best_estimator_.predict(x_val)

print(f'F1-macro val: {f1_score(y_val, y_pred_val, average="macro"):.4f}\n')
print(classification_report(y_val, y_pred_val))

clases = gs.best_estimator_.classes_
cm = confusion_matrix(y_val, y_pred_val, labels=clases)
fig, ax = plt.subplots(figsize=(8,6))
ConfusionMatrixDisplay(cm, display_labels=clases).plot(ax=ax, colorbar=False, cmap='Blues')
plt.title('Matriz de confusion - Val')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

val['pred'] = y_pred_val
errores = val[val[label_col] != val['pred']]
print(f'\nErrores: {len(errores)}/{len(val)} ({len(errores)/len(val)*100:.1f}%)')
print(errores.groupby([label_col,'pred']).size().sort_values(ascending=False).head(8))

## 8. Ensemble (Soft Voting)

In [ ]:
best_ngram = gs.best_params_['tfidf__ngram_range']
best_maxf  = gs.best_params_['tfidf__max_features']
best_c     = gs.best_params_['clf__C']

tfidf_ens = TfidfVectorizer(ngram_range=best_ngram, max_features=best_maxf, sublinear_tf=True, min_df=2, max_df=0.95)
x_tr_ens = tfidf_ens.fit_transform(x_train)
x_va_ens = tfidf_ens.transform(x_val)

clf_lr  = LogisticRegression(C=best_c, max_iter=1000, class_weight='balanced', solver='lbfgs', multi_class='multinomial', random_state=seed)
clf_svc = CalibratedClassifierCV(LinearSVC(C=1, class_weight='balanced', max_iter=3000, random_state=seed))
clf_nb  = ComplementNB(alpha=0.1)

ens = VotingClassifier([('lr',clf_lr),('svc',clf_svc),('nb',clf_nb)], voting='soft')
ens.fit(x_tr_ens, y_train)
f1_ens = f1_score(y_val, ens.predict(x_va_ens), average='macro')
print(f'Ensemble F1-macro val: {f1_ens:.4f}')

## 9. Mejoras — Stemming

In [ ]:
from nltk.stem import SnowballStemmer
nltk.download('punkt', quiet=True)

stemmer = SnowballStemmer('spanish')

def preprocess_stem(text: str) -> str:
    texto = preprocess(text)
    tokens = [stemmer.stem(t) for t in texto.split()]
    return ' '.join(tokens)

train['texto_stem'] = train[text_col].apply(preprocess_stem)
val['texto_stem']   = val[text_col].apply(preprocess_stem)

x_train_stem = train['texto_stem'].values
x_val_stem   = val['texto_stem'].values

pipe_stem = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=80_000, sublinear_tf=True, min_df=2, max_df=0.95)),
    ('clf',   LogisticRegression(C=10, max_iter=1000, class_weight='balanced',
                                  solver='lbfgs', multi_class='multinomial', random_state=seed))
])
pipe_stem.fit(x_train_stem, y_train)
f1_stem = f1_score(y_val, pipe_stem.predict(x_val_stem), average='macro')
print(f'Stemming + TF-IDF  ->  F1-macro val = {f1_stem:.4f}')
print(f'Baseline (sin stem) ->  F1-macro val = {resultados["LogReg C=10"][0]:.4f}')

## 10. Mejoras — Features manuales

In [ ]:
from sklearn.preprocessing import StandardScaler

def features_manuales(df):
    """Extrae features numéricas del texto crudo."""
    feats = pd.DataFrame()
    feats['n_palabras']    = df[text_col].str.split().str.len()
    feats['n_chars']       = df[text_col].str.len()
    feats['long_prom_pal'] = feats['n_chars'] / (feats['n_palabras'] + 1)
    feats['n_mayusculas']  = df[text_col].str.count(r'[A-ZÁÉÍÓÚ]')
    feats['prop_mayusc']   = feats['n_mayusculas'] / (feats['n_chars'] + 1)
    feats['n_signos']      = df[text_col].str.count(r'[.,;:()/]')
    feats['n_numeros']     = df[text_col].str.count(r'\d')
    feats['prop_numeros']  = feats['n_numeros'] / (feats['n_chars'] + 1)
    return feats.values

feats_tr = features_manuales(train)
feats_va = features_manuales(val)

scaler = StandardScaler()
feats_tr_sc = scaler.fit_transform(feats_tr)
feats_va_sc = scaler.transform(feats_va)

# Combinar TF-IDF + features manuales
from scipy.sparse import hstack, csr_matrix

tfidf_fm = TfidfVectorizer(ngram_range=(1,2), max_features=80_000, sublinear_tf=True, min_df=2, max_df=0.95)
x_tr_fm = hstack([tfidf_fm.fit_transform(x_train), csr_matrix(feats_tr_sc)])
x_va_fm = hstack([tfidf_fm.transform(x_val),       csr_matrix(feats_va_sc)])

clf_fm = LogisticRegression(C=10, max_iter=1000, class_weight='balanced',
                             solver='lbfgs', multi_class='multinomial', random_state=seed)
clf_fm.fit(x_tr_fm, y_train)
f1_fm = f1_score(y_val, clf_fm.predict(x_va_fm), average='macro')
print(f'TF-IDF + features manuales  ->  F1-macro val = {f1_fm:.4f}')

# Visualizar importancia de features manuales
nombres_feats = ['n_palabras','n_chars','long_prom_pal','n_mayusculas','prop_mayusc','n_signos','n_numeros','prop_numeros']
df_feats = pd.DataFrame({'feature': nombres_feats, 'media': feats_tr.mean(axis=0), 'std': feats_tr.std(axis=0)})
print('\nEstadísticas de features manuales:')
print(df_feats.round(2))

## 11. Mejoras — FastText entrenado en el corpus

In [ ]:
!pip install -q gensim
from gensim.models import FastText
from gensim.utils import simple_preprocess

# Tokenizamos train + val para entrenar FastText (NUNCA test)
corpus_tokens = [
    simple_preprocess(t, deacc=False)
    for t in list(train['texto_proc'].values) + list(val['texto_proc'].values)
]

ft_model = FastText(
    sentences=corpus_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    epochs=15,
    seed=seed,
    workers=2
)
print(f'Vocabulario FastText: {len(ft_model.wv)} tokens')

def texto_a_vector(texto, modelo):
    """Promedio de vectores de palabras (document embedding)."""
    tokens = simple_preprocess(texto, deacc=False)
    vecs = [modelo.wv[t] for t in tokens if t in modelo.wv]
    if not vecs:
        return np.zeros(modelo.vector_size)
    return np.mean(vecs, axis=0)

x_tr_ft = np.array([texto_a_vector(t, ft_model) for t in x_train])
x_va_ft = np.array([texto_a_vector(t, ft_model) for t in x_val])
print(f'Shape embeddings train: {x_tr_ft.shape}')

clf_ft = LogisticRegression(C=5, max_iter=1000, class_weight='balanced',
                             solver='lbfgs', multi_class='multinomial', random_state=seed)
clf_ft.fit(x_tr_ft, y_train)
f1_ft = f1_score(y_val, clf_ft.predict(x_va_ft), average='macro')
print(f'\nFastText solo         ->  F1-macro val = {f1_ft:.4f}')

In [ ]:
# Combinar TF-IDF + FastText
tfidf_ft = TfidfVectorizer(ngram_range=(1,2), max_features=80_000, sublinear_tf=True, min_df=2, max_df=0.95)
x_tr_tfidf = tfidf_ft.fit_transform(x_train)
x_va_tfidf = tfidf_ft.transform(x_val)

x_tr_combo_ft = hstack([x_tr_tfidf, csr_matrix(x_tr_ft)])
x_va_combo_ft = hstack([x_va_tfidf, csr_matrix(x_va_ft)])

clf_combo_ft = LogisticRegression(C=10, max_iter=1000, class_weight='balanced',
                                   solver='lbfgs', multi_class='multinomial', random_state=seed)
clf_combo_ft.fit(x_tr_combo_ft, y_train)
f1_combo_ft = f1_score(y_val, clf_combo_ft.predict(x_va_combo_ft), average='macro')
print(f'TF-IDF + FastText  ->  F1-macro val = {f1_combo_ft:.4f}')

## 12. Mejoras — Stacking (meta-clasificador)

In [ ]:
from sklearn.ensemble import StackingClassifier

# Usamos el vectorizador del mejor GridSearch
best_ngram = gs.best_params_['tfidf__ngram_range']
best_maxf  = gs.best_params_['tfidf__max_features']
best_c     = gs.best_params_['clf__C']

tfidf_stack = TfidfVectorizer(ngram_range=best_ngram, max_features=best_maxf,
                               sublinear_tf=True, min_df=2, max_df=0.95)
x_tr_stack = tfidf_stack.fit_transform(x_train)
x_va_stack = tfidf_stack.transform(x_val)

estimadores = [
    ('lr',  LogisticRegression(C=best_c, max_iter=1000, class_weight='balanced',
                                solver='lbfgs', multi_class='multinomial', random_state=seed)),
    ('svc', CalibratedClassifierCV(LinearSVC(C=1, class_weight='balanced', max_iter=3000, random_state=seed))),
    ('nb',  ComplementNB(alpha=0.1)),
]
meta_clf = LogisticRegression(C=1, max_iter=500, random_state=seed)

stacking = StackingClassifier(
    estimators=estimadores,
    final_estimator=meta_clf,
    cv=5,
    passthrough=False,
    n_jobs=-1
)
stacking.fit(x_tr_stack, y_train)
f1_stack = f1_score(y_val, stacking.predict(x_va_stack), average='macro')
print(f'Stacking  ->  F1-macro val = {f1_stack:.4f}')

## 13. Resumen comparativo de todas las estrategias

In [ ]:
# Recopila todos los resultados
resumen = {
    'Baseline LogReg C=10':          resultados['LogReg C=10'][0],
    'TF-IDF word+char':              f1_combo,
    'GridSearch mejor':              gs.best_score_,
    'Stemming':                      f1_stem,
    'Features manuales':             f1_fm,
    'FastText solo':                 f1_ft,
    'TF-IDF + FastText':             f1_combo_ft,
    'Ensemble Soft Voting':          f1_ens,
    'Stacking':                      f1_stack,
}

df_resumen = pd.DataFrame(list(resumen.items()), columns=['Estrategia','F1-macro val'])
df_resumen = df_resumen.sort_values('F1-macro val', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(df_resumen['Estrategia'], df_resumen['F1-macro val'], color='navy', edgecolor='white')
for bar, val_f1 in zip(bars, df_resumen['F1-macro val']):
    ax.text(val_f1 + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val_f1:.4f}', va='center', fontsize=9)
ax.set_xlabel('F1-macro val')
ax.set_title('Comparación de todas las estrategias')
ax.set_xlim(0.75, 0.95)
ax.axvline(x=0.88098, color='red', linestyle='--', alpha=0.7, label='Líder leaderboard (0.881)')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

mejor_estrategia = df_resumen.iloc[-1]
print(f'\nMejor estrategia: {mejor_estrategia["Estrategia"]}  ->  {mejor_estrategia["F1-macro val"]:.4f}')

## 14. Entrenamiento final y submission

In [ ]:
# Entrena sobre train + val con el mejor pipeline
modelo_final = gs.best_estimator_
modelo_final.fit(x_all, y_all)
print('Modelo entrenado sobre train + val')

# Preprocesar test SOLO aqui
test['texto_proc'] = test[text_col].apply(preprocess)
y_pred_test = modelo_final.predict(test['texto_proc'].values)

submission = pd.DataFrame({id_col: test[id_col], label_col: y_pred_test})
submission.to_csv('submission.csv', index=False)

print(f'\nDistribucion predicciones:')
print(submission[label_col].value_counts())
submission.head()

In [ ]:
from google.colab import files
files.download('submission.csv')
print('submission.csv listo para subir a Kaggle')

## 15. Referencias

Los siguientes recursos sustentan las decisiones metodológicas de este notebook:

### Representación de texto

- Salton, G., & Buckley, C. (1988). Term-weighting approaches in automatic text retrieval. *Information Processing & Management*, 24(5), 513–523. https://doi.org/10.1016/0306-4573(88)90021-0  
  *(Base teórica de TF-IDF y sublinear_tf)*

- Manning, C. D., Raghavan, P., & Schütze, H. (2008). *Introduction to Information Retrieval*. Cambridge University Press. https://nlp.stanford.edu/IR-book/  
  *(Capítulos 6 y 13: vectorización, n-gramas y clasificación de texto)*

- Zheng, A., & Casari, A. (2018). *Feature Engineering for Machine Learning: Principles and Techniques for Data Scientists*. O'Reilly Media.  
  *(Capítulos 3 y 4: representación de texto, TF-IDF y bag-of-words para clasificación)*

### Clasificadores

- Rennie, J. D. M., Shih, L., Teevan, J., & Karger, D. R. (2003). Tackling the poor assumptions of Naive Bayes text classifiers. *Proceedings of ICML 2003*, 616–623.  
  *(Fundamenta ComplementNB, diseñado para clases desbalanceadas)*

- Fan, R.-E., Chang, K.-W., Hsieh, C.-J., Wang, X.-R., & Lin, C.-J. (2008). LIBLINEAR: A library for large linear classification. *Journal of Machine Learning Research*, 9, 1871–1874.  
  *(Base de LinearSVC y LogisticRegression con solver lbfgs)*

- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3rd ed.). O'Reilly Media.  
  *(Capítulos 2 y 6: pipelines, GridSearchCV, clasificadores y ajuste de hiperparámetros con scikit-learn)*

### Evaluación

- Sokolova, M., & Lapalme, G. (2009). A systematic analysis of performance measures for classification tasks. *Information Processing & Management*, 45(4), 427–437. https://doi.org/10.1016/j.ipm.2009.03.002  
  *(Justifica el uso de F1-macro para clases desbalanceadas)*

### Preprocesamiento y PLN en español

- Bird, S., Klein, E., & Loper, E. (2009). *Natural Language Processing with Python*. O'Reilly Media. https://www.nltk.org/book/  
  *(Stopwords, tokenización y preprocesamiento con NLTK)*

- Jurafsky, D., & Martin, J. H. (2024). *Speech and Language Processing* (3rd ed. draft). https://web.stanford.edu/~jurafsky/slp3/  
  *(Capítulos 4 y 6: clasificación de texto y regresión logística)*

- Bengfort, R., Bilbro, R., & Ojeda, T. (2018). *Applied Text Analysis with Python: Enabling Language-Aware Data Products with Machine Learning*. O'Reilly Media.  
  *(Pipeline completo de clasificación de texto con scikit-learn: vectorización, modelos y evaluación)*

### Ensambles

- Dietterich, T. G. (2000). Ensemble methods in machine learning. *Lecture Notes in Computer Science*, 1857, 1–15. https://doi.org/10.1007/3-540-45014-9_1  
  *(Fundamento teórico del Soft Voting Ensemble)*

### Análisis y manipulación de datos

- McKinney, W. (2022). *Python for Data Analysis: Data Wrangling with pandas, NumPy, and Jupyter* (3rd ed.). O'Reilly Media. https://wesmckinney.com/book/  
  *(Manipulación de datos con pandas y numpy a lo largo del notebook)*

### Scikit-learn (implementación)

- Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html